# 02 · Embeddings와 유사도

**목표**: 텍스트를 숫자 벡터로 바꾸고 코사인 유사도로 의미상 가까운 문장을 비교합니다.

임베딩은 RAG와 grounding 검색의 기반입니다. 질문과 문서 조각을 같은 벡터 공간에 놓고 가까운 문서를 찾습니다. 실행 전 루트 `.env`의 `EMBED_MODEL`을 현재 AI Core에 배포된 모델명으로 설정하세요. 이 실습은 입력 텍스트의 임베딩과 결과 조회만 수행하며 외부 업무 상태를 변경하지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 두 텍스트의 의미적 거리 비교 (UI)</h2>
<p style="margin: 7px 0;">아래 UI에 두 텍스트를 입력하고 Compare를 누르세요. embedding 모델은 루트 <code>.env</code>의 <code>EMBED_MODEL</code>을 자동으로 사용합니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>Text A와 Text B에 비교할 텍스트를 입력합니다.</li><li>표시된 <code>Embedding model</code>이 현재 <code>.env</code> 설정과 일치하는지 확인합니다.</li><li>오른쪽 Result에서 <code>cosine_similarity</code>가 1에 가까울수록 두 벡터의 방향이 비슷한지 확인합니다.</li></ol>
<p style="margin: 7px 0;">UI는 조회용입니다. 임베딩 요청만 보내며 업무 데이터나 외부 상태를 수정하지 않습니다.</p>
  </div>
</div>

In [4]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "shared").exists())
for module_path in (project_root / "shared", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.gradio_ui import build_embedding_ui
from src.lab_core import compare_embeddings

embedding_ui = build_embedding_ui(compare_embeddings)
embedding_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=680)

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>두 텍스트를 입력했을 때 모델명, vector 차원, 비교 결과가 JSON으로 보인다.</li><li>텍스트를 비우면 SDK 호출 전에 입력 오류가 표시된다.</li><li>UI 결과는 검증된 코어 API의 반환값이며, 상태 변경은 발생하지 않는다.</li></ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 같은 코어 API를 Python으로 살펴보는 선택 심화 자료입니다.</p>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 호출하는 <code>compare_embeddings()</code>를 사용합니다. SDK 호출, 입력 검증, 벡터 차원 확인, cosine similarity 계산은 모두 코어에 있으므로 노트북이 같은 로직을 다시 구현하지 않습니다.</p>
  </div>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "shared").exists())
for module_path in (project_root / "shared", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.lab_core import compare_embeddings

settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 임베딩 생성과 쌍별 비교</h2>
<p style="margin: 7px 0;">텍스트 목록을 한 번에 전달하면 코어가 한 번의 embedding 요청으로 벡터를 받고 모든 텍스트 쌍의 cosine similarity를 반환합니다. 같은 텍스트가 중복되어도 비교 결과가 덮어써지지 않도록 결과는 목록으로 표현합니다.</p>
  </div>
</div>

In [ ]:
examples = ["apple", "orange", "phone", "I love dogs", "I love animals", "I hate cats"]
comparison = compare_embeddings(examples, settings)
comparison

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 반환된 근거 읽기</h2>
<p style="margin: 7px 0;">차원 수와 이미 계산된 비교 결과를 확인합니다. 여기서는 벡터나 cosine 함수를 다시 만들지 않고 코어가 반환한 값을 그대로 읽습니다. 일반적으로 의미가 가까운 쌍의 상대적 순서를 비교하는 것이 유용합니다.</p>
  </div>
</div>

In [ ]:
print(f"embedding model: {comparison['model']}")
print(f"vector dimensions: {comparison['dimensions']}")
for item in comparison["comparisons"]:
    print(f"{item['left']} ↔ {item['right']}: {item['cosine_similarity']:.3f}")

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>반환값에서 embedding 모델과 vector 차원 수를 확인했다.</li><li><code>comparisons</code>의 각 항목이 좌·우 텍스트와 계산된 cosine similarity를 함께 가진다는 것을 확인했다.</li><li>유사도 계산과 입력 검증은 노트북이 아닌 테스트된 코어에 있다는 것을 확인했다.</li></ul>
  </div>
</div>